In [1]:
project_root_path = '${PROJECT_ROOT_PATH}'
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

import sys
import os
import time
import json
import multiprocessing as mp
import IPython

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend

sys.path.append(project_root_path)
import lib.launchit # @launchit.disable
import lib.module_launcher # @launchit.disable

In [2]:
# @launchit.disable
with open(get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as cf:
    notebook_fname = json.load(cf)['jupyter_session']
    notebook_basename = os.path.basename(notebook_fname)
    notebook_name, notebook_ext = os.path.splitext(notebook_basename)

notebook_fname

'/home/misha/dev/mine/neurovision/experiment/optuna/optuna_basic_multiproc_test.ipynb'

In [3]:
STUDY_NAME = 'optuna_basic_miltiproc_test'

RUN_DIR = os.path.join(project_root_path, 'run', 'launchit', 'experiment', 'optuna')
os.makedirs(RUN_DIR, exist_ok=True)

STUDY_FNAME = os.path.join(RUN_DIR, STUDY_NAME + '.log')

In [4]:
def objective(trial):
    print(f"Running trial.number={trial.number} in process {os.getpid()}")
    x = trial.suggest_float("x", -10, 10)
    # time.sleep(10)
    return (x - 2) ** 2

def run_optimization(params):
    print(f'params={params}')
    study = optuna.create_study(
        study_name=STUDY_NAME,
        storage=JournalStorage(JournalFileBackend(file_path=STUDY_FNAME)),
        load_if_exists=True, # Useful for multi-process or multi-node optimization.
    )
    study.optimize(objective, n_trials=1)

In [5]:
# @launchit.disable
launch_fname = lib.launchit.launchit(notebook_fname, make_py_file=True, expandvars={'PROJECT_ROOT_PATH': project_root_path}, dir_name=RUN_DIR, verbosity=0)
mp_ctx = mp.get_context('spawn')

with mp_ctx.Pool(processes=4) as pool:
    lp_list = [lib.module_launcher.LaunchParameters(module_fname=launch_fname, func_name='run_optimization', func_args=i, verbosity=3) for i in range(10)]
    pool.map(lib.module_launcher.launch, lp_list)

lp.module_fname='/home/misha/dev/mine/neurovision/run/launchit/experiment/optuna/optuna_basic_multiproc_test-launch1.py', module_dir_name='/home/misha/dev/mine/neurovision/run/launchit/experiment/optuna', module_name='optuna_basic_multiproc_test-launch1', lp.func_name='run_optimization'
Running optuna_basic_multiproc_test-launch1.run_optimization0
lp.module_fname='/home/misha/dev/mine/neurovision/run/launchit/experiment/optuna/optuna_basic_multiproc_test-launch1.py', module_dir_name='/home/misha/dev/mine/neurovision/run/launchit/experiment/optuna', module_name='optuna_basic_multiproc_test-launch1', lp.func_name='run_optimization'
Running optuna_basic_multiproc_test-launch1.run_optimization2
lp.module_fname='/home/misha/dev/mine/neurovision/run/launchit/experiment/optuna/optuna_basic_multiproc_test-launch1.py', module_dir_name='/home/misha/dev/mine/neurovision/run/launchit/experiment/optuna', module_name='optuna_basic_multiproc_test-launch1', lp.func_name='run_optimization'
Running optu

[I 2026-01-23 17:14:45,698] A new study created in Journal with name: optuna_basic_miltiproc_test
[I 2026-01-23 17:14:45,710] Trial 0 finished with value: 52.359570592933274 and parameters: {'x': 9.235991334498216}. Best is trial 0 with value: 52.359570592933274.
[I 2026-01-23 17:14:45,711] Using an existing study with name 'optuna_basic_miltiproc_test' instead of creating a new one.
[I 2026-01-23 17:14:45,715] Trial 1 finished with value: 12.590556367331025 and parameters: {'x': 5.548317399462881}. Best is trial 1 with value: 12.590556367331025.
[I 2026-01-23 17:14:45,717] Using an existing study with name 'optuna_basic_miltiproc_test' instead of creating a new one.
[I 2026-01-23 17:14:45,720] Trial 2 finished with value: 0.4013868229168664 and parameters: {'x': 1.3664490368432336}. Best is trial 2 with value: 0.4013868229168664.
[I 2026-01-23 17:14:45,722] Using an existing study with name 'optuna_basic_miltiproc_test' instead of creating a new one.
[I 2026-01-23 17:14:45,726] Trial 